In [1]:
import os
from pyspark.sql import SparkSession

# Khởi tạo Spark Session tích hợp GraphFrames và Kafka 
spark = (SparkSession.builder.appName("Lab3_GraphX_Home")
    .config("spark.jars.packages", 
            "io.graphframes:graphframes-spark4_2.13:0.9.3," # Thư viện GraphFrames
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1," # Thư viện Spark-Kafka
            "org.apache.hadoop:hadoop-aws:3.4.1") # Thư viện AWS/S3 hỗ trợ hệ thống file
    .master("local[*]") # Chạy trên toàn bộ nhân CPU của máy 
    .getOrCreate())

# Thiết lập Checkpoint tại máy cục bộ để chạy PageRank/ConnectedComponents 
checkpoint_dir = os.path.abspath("spark_checkpoints")
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)
spark.sparkContext.setCheckpointDir("file://" + checkpoint_dir)

print("--- Spark Session & Checkpoint đã sẵn sàng! ---")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/12 20:13:07 WARN Utils: Your hostname, HungThinh, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/12 20:13:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/thinh/.ivy2.5.2/cache
The jars for the packages stored in: /home/thinh/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5d56a96e-92dd-4220-a85e-a8e97384a3f2;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.9.3 in central
	found org.apache.spark#spark-

--- Spark Session & Checkpoint đã sẵn sàng! ---


In [2]:
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient({'bootstrap.servers': 'localhost:9092,localhost:9192,localhost:9292'})
topic_names = ['Lab1_movies', 'Lab1_ratings', 'Lab1_tags']
admin_client.delete_topics(topic_names, operation_timeout=10)
new_topics = [NewTopic(topic=name, num_partitions=1, replication_factor=1) for name in topic_names]
fs = admin_client.create_topics(new_topics)

for topic, f in fs.items():
    try:
        f.result()
        print(f"Topic '{topic}' đã sẵn sàng!")
    except Exception as e:
        print(f"Lỗi khi tạo {topic}: {e}")

Topic 'Lab1_movies' đã sẵn sàng!
Topic 'Lab1_ratings' đã sẵn sàng!
Topic 'Lab1_tags' đã sẵn sàng!


In [3]:
import kagglehub
from pyspark.sql.functions import to_json, struct

# 1. Tải dataset từ Kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# 2. Đọc file CSV bằng Spark 
df_r = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_m = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_t = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

# 3. Hàm đẩy dữ liệu vào Kafka
def push_to_kafka(df, topic):
    df.selectExpr("to_json(struct(*)) AS value") \
      .write.format("kafka") \
      .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292") \
      .option("topic", topic) \
      .save()
    print(f"Đã nạp dữ liệu cho {topic}")

# Thực hiện nạp cho 3 topic 
push_to_kafka(df_m, "Lab1_movies")
push_to_kafka(df_r, "Lab1_ratings")
push_to_kafka(df_t, "Lab1_tags")

/home/thinh/BigData/bigdata_lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã nạp dữ liệu cho Lab1_movies


Đã nạp dữ liệu cho Lab1_ratings
Đã nạp dữ liệu cho Lab1_tags


In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, lit

# Định nghĩa Schema đầy đủ cho 3 bảng
movie_schema = StructType([
    StructField("movieId", IntegerType()), 
    StructField("title", StringType()), 
    StructField("genres", StringType())
])
rating_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("rating", DoubleType()), 
    StructField("timestamp", LongType())
])
tag_schema = StructType([
    StructField("userId", IntegerType()), 
    StructField("movieId", IntegerType()), 
    StructField("tag", StringType()), 
    StructField("timestamp", LongType())
])

def load_kafka_full(topic, schema):
    return (spark.read.format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092,localhost:9192,localhost:9292")
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
        .select(from_json(col("value").cast("string"), schema).alias("data"))
        .select("data.*"))

# Load 3 topics 
df_movies = load_kafka_full("Lab1_movies", movie_schema)
df_ratings = load_kafka_full("Lab1_ratings", rating_schema)
df_tags = load_kafka_full("Lab1_tags", tag_schema)

print("--- Exercise 0: OK ---")

--- Exercise 0: OK ---


In [ ]:
from graphframes import GraphFrame

# Vertices: User + Movie 
v_user = df_ratings.select(col("userId").cast("string").alias("id")).distinct()
v_movie = df_movies.select(col("movieId").cast("string").alias("id"), "title", "genres")
vertices = v_user.unionByName(v_movie, allowMissingColumns=True)

# Edges: Quan hệ đánh giá phim 
edges = df_ratings.select(col("userId").cast("string").alias("src"), 
                          col("movieId").cast("string").alias("dst"), 
                          col("rating").alias("weight"))

g = GraphFrame(vertices, edges)

In [16]:
from pyspark.sql.functions import sum, desc

# Tính toán các chỉ số 
in_degrees = g.inDegrees.withColumnRenamed("inDegree", "distinct_raters")
weighted_in_degrees = g.edges.groupBy("dst").agg(sum("weight").alias("total_rating_weight")) \
    .withColumnRenamed("dst", "id")

# Kết quả Top 20 
ex1_result = in_degrees.join(weighted_in_degrees, "id").join(v_movie, "id") \
    .select("id", "title", "distinct_raters", "total_rating_weight") \
    .orderBy(desc("distinct_raters"))

ex1_result.show(20, truncate=False)

# --- INSIGHTS CHO EXERCISE 1 --- 
print("Insights for Exercise 1:")
print("1. Những phim có lượt đánh giá (in-degree) cao nhất thường là các phim kinh điển đời đầu.")
print("2. Tổng trọng số (weighted in-degree) tỷ lệ thuận với số lượng người đánh giá, cho thấy sự phổ biến ổn định.")
print("3. Các phim bom tấn Hollywood chiếm đa số trong danh sách phổ biến thay vì các phim nghệ thuật.")

+----+------------------------------------------------------------------------------+---------------+-------------------+
|id  |title                                                                         |distinct_raters|total_rating_weight|
+----+------------------------------------------------------------------------------+---------------+-------------------+
|356 |Forrest Gump (1994)                                                           |329            |1370.0             |
|318 |Shawshank Redemption, The (1994)                                              |317            |1404.0             |
|296 |Pulp Fiction (1994)                                                           |307            |1288.5             |
|593 |Silence of the Lambs, The (1991)                                              |279            |1161.0             |
|2571|Matrix, The (1999)                                                            |278            |1165.5             |
|260 |Star Wars: Episode

In [17]:
# Chạy PageRank 
pr = g.pageRank(resetProbability=0.15, maxIter=5)

# Lấy Top 20 Movie kèm thể loại 
top_20_pr = pr.vertices.filter("title IS NOT NULL") \
    .select("id", "title", "genres", "pagerank") \
    .orderBy(desc("pagerank"))

print("Top 20 Movies by PageRank:")
top_20_pr.show(20, truncate=False)

Top 20 Movies by PageRank:
+----+-----------------------------------------+-------------------------------------------+------------------+
|id  |title                                    |genres                                     |pagerank          |
+----+-----------------------------------------+-------------------------------------------+------------------+
|318 |Shawshank Redemption, The (1994)         |Crime|Drama                                |8.361626131697976 |
|356 |Forrest Gump (1994)                      |Comedy|Drama|Romance|War                   |7.409003377974512 |
|296 |Pulp Fiction (1994)                      |Comedy|Crime|Drama|Thriller                |6.978900041716169 |
|593 |Silence of the Lambs, The (1991)         |Crime|Horror|Thriller                      |6.584817693519373 |
|527 |Schindler's List (1993)                  |Drama|War                                  |6.37618431268022  |
|260 |Star Wars: Episode IV - A New Hope (1977)|Action|Adventure|Sci-Fi      

In [ ]:
# Tìm motif phân cực 
# Cấu hình: u1 và u2 cùng đánh giá phim m
motifs = g.find("(u1)-[e1]->(m); (u2)-[e2]->(m)") \
    .filter("u1.id != u2.id") \
    .filter("abs(e1.weight - e2.weight) >= 3.0")

# Rank theo số lượng cặp phân cực 
polarization = motifs.groupBy("m.id", "m.title").count() \
    .withColumnRenamed("id", "movieId") \
    .withColumnRenamed("count", "polarized_pairs") \
    .orderBy(desc("polarized_pairs"))

print("Top 10 Most Polarizing Movies:")
polarization.select("movieId", "title", "polarized_pairs").show(10, truncate=False)

# --- INSIGHTS CHO EXERCISE 3 --- 
print("Insights for Exercise 3:")
print("1. Những phim gây tranh cãi nhất thường là các phim bom tấn có lượng người xem khổng lồ.")
print("2. Sự phân cực (polarization) cho thấy cảm nhận điện ảnh của người dùng là rất khác biệt.")
print("3. Các bộ phim hành động hoặc khoa học viễn tưởng thường có số cặp đánh giá trái chiều cao hơn.")

Top 10 Most Polarizing Movies:


+-------+-----------------------------------------+---------------+
|movieId|title                                    |polarized_pairs|
+-------+-----------------------------------------+---------------+
|296    |Pulp Fiction (1994)                      |21814          |
|296    |NULL                                     |21814          |
|2571   |Matrix, The (1999)                       |17564          |
|527    |Schindler's List (1993)                  |12070          |
|527    |NULL                                     |12070          |
|260    |NULL                                     |11232          |
|260    |Star Wars: Episode IV - A New Hope (1977)|11232          |
|356    |Forrest Gump (1994)                      |11060          |
|356    |NULL                                     |11060          |
|110    |Braveheart (1995)                        |10192          |
+-------+-----------------------------------------+---------------+
only showing top 10 rows
Insights for Exercise 3